# LoRA-FA: LoRA with a frozen A

LoRA-FA (Zhang et al., 2023) keeps LoRA's structure and **freezes `A` at its random init**. Only `B` is trained:

$$W' = W + \frac{\alpha}{r}\, B A, \qquad A \text{ frozen (random)}, \quad B \text{ trainable, starts at } 0$$

Two savings:

1. **Parameters.** Only `d_out · r` numbers are trained, about half of LoRA's.
2. **Activation memory.** That is the real reason for the method. To get the gradient of `A`, autograd has to keep each layer's input `x`: `d_in` numbers per token. With `A` frozen it only keeps `A x`: `r` numbers per token.

A random `A` is a fixed random projection of the input down to `r` dimensions. `B` learns how to map that projection to a useful weight update.

In [ ]:
import torch

d_out, d_in, r = 800, 500, 4
A = torch.randn(r, d_in) * 0.01
B = torch.zeros(d_out, r)

print(f"full finetuning: {d_out * d_in:,} parameters")
print(f"LoRA (A and B):  {d_out * r + r * d_in:,} parameters")
print(f"LoRA-FA (B):     {d_out} x {r} = {B.numel():,} parameters")

tokens = 256
print(f"saved for backward per token, LoRA:    x   -> {d_in} numbers")
print(f"saved for backward per token, LoRA-FA: A x -> {r} numbers")
print(f"for {tokens} tokens: {tokens * d_in:,} vs {tokens * r:,}")

## Setup

Same base model and tokenizer as `finetuning.ipynb`. Run `shop_assistant_dataset.ipynb` first so `data/instruction-data.json` exists.

In [ ]:
import json
import sys
import time
from functools import partial
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from reasoning_from_scratch.qwen3 import (
    QWEN_CONFIG_06_B,
    KVCache,
    Qwen3Model,
    Qwen3Tokenizer,
    download_qwen3_small,
)

torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    print("No CUDA GPU found. Everything runs on CPU, but training will be slow.")

download_qwen3_small(kind="base", out_dir=ROOT / "qwen3")
tokenizer = Qwen3Tokenizer(ROOT / "qwen3" / "tokenizer-base.json")
model = Qwen3Model(QWEN_CONFIG_06_B)
weights = torch.load(ROOT / "qwen3" / "qwen3-0.6B-base.pth", map_location="cpu", weights_only=True)
model.load_state_dict(weights)
del weights
model.to(device)
model.eval()

n_base = sum(param.numel() for param in model.parameters())
print(f"base model: {n_base / 1e6:.1f}M parameters on {device}")

In [ ]:
INSTRUCTION = (
    "You are a shop assistant for a clothing and shoe store. "
    "Read the customer message and return only the next action as JSON "
    "with keys action, slots, missing, and reply. "
    "Do not invent products, prices, or stock."
)


def format_input(entry: dict) -> str:
    """Format an instruction entry as the training prompt."""
    instruction_text = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text + input_text


class InstructionDataset(Dataset):
    """Prompt plus response, with the prompt length kept for the loss mask."""

    def __init__(self, data: list[dict], tokenizer) -> None:
        self.examples = []
        for entry in data:
            prompt = format_input(entry) + "\n\n### Response:\n"
            full = prompt + entry["output"]
            prompt_ids = tokenizer.encode(prompt)
            full_ids = tokenizer.encode(full)
            self.examples.append((full_ids, len(prompt_ids)))

    def __getitem__(self, index: int) -> tuple[list[int], int]:
        return self.examples[index]

    def __len__(self) -> int:
        return len(self.examples)


def collate(batch, device, pad_id: int):
    """Pad a batch and ignore prompt and padding tokens in the loss."""
    sequences, prompt_lens = [], []
    for ids, prompt_len in batch:
        sequences.append(ids + [pad_id])
        prompt_lens.append(prompt_len)
    max_len = max(len(seq) for seq in sequences)
    input_rows, target_rows = [], []
    for seq, prompt_len in zip(sequences, prompt_lens):
        padded = seq + [pad_id] * (max_len - len(seq))
        inputs = torch.tensor(padded[:-1], device=device)
        targets = torch.tensor(padded[1:], device=device)
        targets[: prompt_len - 1] = -100
        pad_positions = (targets == pad_id).nonzero(as_tuple=True)[0]
        if len(pad_positions) > 1:
            targets[pad_positions[1:]] = -100
        input_rows.append(inputs)
        target_rows.append(targets)
    return torch.stack(input_rows), torch.stack(target_rows)


with open(ROOT / "data" / "instruction-data.json", encoding="utf-8") as f:
    data = json.load(f)

train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.1)
train_data = data[:train_portion]
test_data = data[train_portion : train_portion + test_portion]
val_data = data[train_portion + test_portion :]

collate_fn = partial(collate, device=device, pad_id=tokenizer.pad_token_id)
train_loader = DataLoader(
    InstructionDataset(train_data, tokenizer),
    batch_size=1,
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=0,
)
val_loader = DataLoader(
    InstructionDataset(val_data, tokenizer),
    batch_size=1,
    collate_fn=collate_fn,
    shuffle=False,
    num_workers=0,
)
print("train", len(train_data), "val", len(val_data), "test", len(test_data))

In [ ]:
# Every linear layer inside the 28 transformer blocks gets an adapter.
# The token embedding, the norms, and the output head stay as they are.
TARGETS = {"att": ("W_query", "W_key", "W_value", "out_proj"), "ff": ("fc1", "fc2", "fc3")}


def target_layers(model):
    """Yield (parent module, attribute name) for every adapted linear layer."""
    for block in model.trf_blocks:
        for parent_name, names in TARGETS.items():
            parent = getattr(block, parent_name)
            for name in names:
                yield parent, name


def add_adapters(model, wrap) -> None:
    """Freeze every base weight, then replace each target layer with wrap(layer)."""
    for param in model.parameters():
        param.requires_grad = False
    for parent, name in target_layers(model):
        setattr(parent, name, wrap(getattr(parent, name)))


def count_trainable(model) -> int:
    """Count trainable parameters. Shared parameters are counted once."""
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


@torch.no_grad()
def evaluate_loss(model, loader) -> float:
    """Mean masked cross-entropy over a loader."""
    model.eval()
    losses = []
    for inputs, targets in loader:
        logits = model(inputs)
        loss = nn.functional.cross_entropy(logits.flatten(0, 1).float(), targets.flatten())
        losses.append(loss.item())
    return sum(losses) / len(losses)


def train(model, params, lr: float, accum_steps: int = 4, weight_decay: float = 0.0, log_every: int = 40) -> None:
    """One pass over the training split. Gradients from 4 examples make one AdamW step."""
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    model.train()
    optimizer.zero_grad()
    start = time.time()
    for step, (inputs, targets) in enumerate(train_loader, start=1):
        logits = model(inputs)
        loss = nn.functional.cross_entropy(logits.flatten(0, 1).float(), targets.flatten())
        (loss / accum_steps).backward()
        if step % accum_steps == 0 or step == len(train_loader):
            optimizer.step()
            optimizer.zero_grad()
        if step % log_every == 0 or step == len(train_loader):
            print(f"step {step}/{len(train_loader)} loss {loss.item():.3f}")
    model.eval()
    print(f"training took {time.time() - start:.0f}s")
    if device.type == "cuda":
        print(f"peak GPU memory {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB")


def prompt_for(customer: str) -> str:
    """Wrap a customer message in the training prompt."""
    return format_input({"instruction": INSTRUCTION, "input": customer}) + "\n\n### Response:\n"


def generate(model, prompt: str, max_new_tokens: int = 100) -> str:
    """Greedy decoding with a KV cache. Stops at <|endoftext|>."""
    token_ids = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    new_ids = []
    with torch.inference_mode():
        logits = model(token_ids, cache=cache)
        for _ in range(max_new_tokens):
            next_id = logits[:, -1].argmax(dim=-1, keepdim=True)
            if next_id.item() == tokenizer.eos_token_id:
                break
            new_ids.append(next_id.item())
            logits = model(next_id, cache=cache)
    model.reset_kv_cache()
    return tokenizer.decode(new_ids).strip()


def score_actions(model, rows: list[dict]) -> None:
    """Generate an action for each row and compare it with the expected one."""
    valid = correct = 0
    for entry in rows:
        text = generate(model, prompt_for(entry["input"]))
        try:
            payload = json.loads(text)
        except json.JSONDecodeError:
            continue
        if not isinstance(payload, dict):
            continue
        valid += 1
        correct += payload.get("action") == json.loads(entry["output"])["action"]
    print(f"valid JSON {valid}/{len(rows)}   right action {correct}/{len(rows)}")


SAMPLE = "Busco tenis para correr, talla 42, que no sean negros"

## The LoRA-FA layer

Same as LoRA, except `A` has `requires_grad=False`. It is still a `Parameter`, so the memory check below can switch it back on for a fair comparison.

In [ ]:
class LoRAFALinear(nn.Module):
    """Frozen linear layer plus (alpha / r) * B @ A, with A frozen at a random init."""

    def __init__(self, base: nn.Linear, rank: int, alpha: float) -> None:
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(rank, base.in_features, device=device), requires_grad=False)
        self.B = nn.Parameter(torch.zeros(base.out_features, rank, device=device))
        nn.init.kaiming_uniform_(self.A, a=5**0.5)
        self.scaling = alpha / rank

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        update = (x @ self.A.to(x.dtype).T).float() @ self.B.T
        return self.base(x) + (update * self.scaling).to(x.dtype)

    def delta_weight(self) -> torch.Tensor:
        """The full d_out x d_in update this adapter represents."""
        return (self.B @ self.A) * self.scaling


RANK, ALPHA = 8, 16
add_adapters(model, lambda layer: LoRAFALinear(layer, RANK, ALPHA))

n_trainable = count_trainable(model)
print(f"trainable {n_trainable:,} / {n_base:,} base parameters ({100 * n_trainable / n_base:.2f}%)")

## Activation memory: LoRA vs LoRA-FA

One forward and backward pass on the longest training example. The first run also trains `A` (plain LoRA), the second keeps it frozen (LoRA-FA). Only the GPU has a peak memory counter, so the cell skips on CPU.

In [ ]:
fa_layers = [getattr(parent, name) for parent, name in target_layers(model)]
longest = max(train_loader.dataset.examples, key=lambda example: len(example[0]))
inputs, targets = collate_fn([longest])


def peak_memory_one_step(train_A: bool) -> float:
    """Peak GPU memory in MiB for one forward and backward pass."""
    for layer in fa_layers:
        layer.A.requires_grad = train_A
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    start = torch.cuda.memory_allocated()
    logits = model(inputs)
    loss = nn.functional.cross_entropy(logits.flatten(0, 1).float(), targets.flatten())
    loss.backward()
    peak = torch.cuda.max_memory_allocated() - start
    model.zero_grad(set_to_none=True)
    for layer in fa_layers:
        layer.A.requires_grad = False
    return peak / 1024**2


if device.type == "cuda":
    print(f"{inputs.shape[1]} tokens")
    lora_mib = peak_memory_one_step(train_A=True)
    fa_mib = peak_memory_one_step(train_A=False)
    print(f"LoRA    (A and B trained): {lora_mib:.0f} MiB above the weights")
    print(f"LoRA-FA (B trained):       {fa_mib:.0f} MiB above the weights")
else:
    print("Skipped: needs a CUDA GPU.")

In [ ]:
val_before = evaluate_loss(model, val_loader)
print(f"validation loss before training: {val_before:.3f}")
print("base model output:")
print(generate(model, prompt_for(SAMPLE)))

## Training

Same loop as `finetuning.ipynb`: one example at a time, the loss only scores the action, and 4 gradients add up before each AdamW step. Only the adapter parameters go to the optimizer.

In [ ]:
fa_params = [param for param in model.parameters() if param.requires_grad]
train(model, fa_params, lr=2e-4)

In [ ]:
val_after = evaluate_loss(model, val_loader)
print(f"validation loss: {val_before:.3f} -> {val_after:.3f}")
print("finetuned output:")
print(generate(model, prompt_for(SAMPLE)))
score_actions(model, test_data)

## Save the adapter, then merge it

Only `B` is trained, but `A` is needed to rebuild `ΔW`. `A` comes from a fixed seed, so storing the seed would be enough. Here both go into the file to keep it simple.

In [ ]:
adapter = {
    name: param.detach().cpu()
    for name, param in model.named_parameters()
    if name.endswith(".A") or name.endswith(".B")
}
adapter_path = ROOT / "qwen3" / "shop-lora-fa-adapter.pth"
torch.save(adapter, adapter_path)
print(f"adapter: {len(adapter)} tensors, {adapter_path.stat().st_size / 1024**2:.1f} MiB")


@torch.no_grad()
def merge_adapters(model) -> None:
    """Add each adapter's update into its base weight and drop the adapter."""
    for parent, name in target_layers(model):
        layer = getattr(parent, name)
        merged = layer.base.weight.float() + layer.delta_weight()
        layer.base.weight.copy_(merged.to(layer.base.weight.dtype))
        setattr(parent, name, layer.base)


merge_adapters(model)
print(generate(model, prompt_for(SAMPLE)))

## Recap

- About half of LoRA's trainable parameters at the same rank, because only `B` learns.
- Less activation memory, because no layer has to keep its full input for `A`'s gradient.
- The update can only use the `r` input directions picked by the random `A`. That limits it more than LoRA. A slightly higher rank makes up for it, and it is still cheap.